# TradeSense · Component 03 · notebook 44: fixing the multi-horizon view (amendment A7; CPU, no training)

**What notebook 42 showed (real run).** The 15-minute and 1-hour roll-ups lose recall against the 1-minute Alert (0.708 / 0.678 versus 0.778), the daily peak score does not react to the 5 August crash (73 / 64), and the hourly Watch tier does not help volume bursts. Writing this down is not the fix. Two causes can actually be repaired:

1. **The tiers were not burden-matched.** Window Alert tiers flagged only 0.83 (15 min) and 0.35 (1 h) genuine windows per day, while the 1-minute Alert raises 1.82 episodes per day. Comparing them is like comparing a strict and a lenient alarm. **Fix:** declare the window tiers by *burden*, in genuine flagged windows per day: Alert = **2 per day**, Watch = **4 per day** (the same budgets as the 1-minute Alert 2/day and Watch-4), for both 15 minutes and 1 hour.
2. **The window score was only the peak.** One high bar says little about a sustained build-up (slow accumulation, momentum ignition lose most at window level). **Fix:** five candidate window scores are compared *on VAL-3 only* (out-of-fold fused scores): `max`, `top3` (mean of the three largest bars), `q90` (90th percentile of the window), `count` (number of bars above the 99th percentile of normal DEV bars), `excess` (sum of the excess above that level). The simplest candidate within 0.02 of the best (mean Alert macro-8 recall at 15 minutes and 1 hour) is chosen.

**Honest status of the data.** CAL-3 was already used once in notebook 42. Amendment A7 is therefore selected on **VAL-3 only**; CAL-3 is then reported as a *second look at a VAL-chosen rule*, disclosed as such, not as a fresh test. TEST remains closed.

**Pre-registered hypotheses (CAL-3, fusion v0).**
- **H-N1:** at 15 minutes and at 1 hour the Alert tier (2 genuine flagged windows per day, chosen window score) reaches macro-8 recall >= the 1-minute Alert (recomputed, 0.778) minus 0.02, and the genuine flagged windows per day stay within 0.5x-2x of 2.
- **H-N2:** the chosen window score has window ROC-AUC >= the peak score at both horizons (window flagged versus genuine windows).
- **H-N3 (day view, exploratory; defined after seeing notebook 42):** the number of hourly Watch windows in a day (a descriptive count, not a score) of the real 5 August 2024 UTC day ranks at or above the 80th percentile of the 30 genuine August days for both symbols. This is one real day and not evidence about manipulation.
- **Not claimed:** volume bursts. They are shown to be ambiguous at 1-minute level (nb40, nb42) and are routed to the trade-level channel (phase 2); this notebook does not pretend that windows fix them, and reports their recall either way.

TEST-3, TEST-B and every TGNN test day are untouched. No frozen rule is changed: the 1-minute Alert and Watch stay exactly as declared.

In [ ]:
import os, sys, json, time, hashlib, warnings, datetime
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 2000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
for f in ['declared_fusion_v0.json', 'fusion_v0_cal3_table.csv', 'declared_watch_v1.json']:
    assert (OUT/f).exists(), f'{f} missing: run notebooks 35 (patched), 40 first'
for s in SYMBOLS: assert (OUT/'scores'/f'fusion_v0_{s}.npz').exists(), 'run notebook 35 first'
sys.path.insert(0, str(OUT)); import importlib; import b2_lib as B; importlib.reload(B)
say('protocol B2 intact, sha256 ' + ph[:16] + '; library sha256 ' + sha_file(OUT/'b2_lib.py') + ('  [MOCK RUN]' if MOCK else ''))
HOR = {'15 min': 15, '1 hour': 60}; BURDEN = {'Alert': 2.0, 'Watch': 4.0}      # genuine flagged windows per day (declared before the run)
KINDS = ['max', 'top3', 'q90', 'count', 'excess']; SIMPLE = {k: i for i, k in enumerate(KINDS)}

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 1b0b46630763c944


## 1 · Data, window scores, burden-matched thresholds

In [ ]:
AOLEN = 31 * 1440; CAL_LO = 1440; DEV_LO, DEV_HI = 90720, 129600; DEVST = DEV_LO + 1440
CALV = B.valid_mask(AOLEN, CAL_LO, AOLEN)
FS = {}; EC = {}; EVV = {}
for s in SYMBOLS:
    z = np.load(OUT/'scores'/f'fusion_v0_{s}.npz'); assert not any('TEST' in k for k in z.files)
    FS[s] = {k: z[k] for k in ['GEN_AO', 'CAL_r0', 'OOF_GEN_L1', 'OOF_VAL_r0', 'OOF_VAL_r1']}
    e = pd.read_csv(L3/f'l3v3_{s}_CAL_events.csv'); EC[s] = e[e.replica == 0].reset_index(drop=True)
    v = pd.read_csv(L3/f'l3v3_{s}_VAL_events.csv'); v = v[(v.t0 >= DEVST) & (v.t0 + v.dur <= DEV_HI)]; EVV[s] = {int(r): g.reset_index(drop=True) for r, g in v.groupby('replica')}
FAMN = B.FAMS; FAMS = list(FAMN)
ECALL = pd.concat([EC[s].assign(sym=s) for s in SYMBOLS], ignore_index=True)
DW = json.load(open(OUT/'declared_watch_v1.json')); QA = DW['alert']['q']
def thr_roll(g, q): t = B.rolling_thr(g, q); return np.where(np.isnan(t), np.inf, t)
def event_hits(a, ev):
    c = np.concatenate([[0], np.cumsum(np.nan_to_num(a).astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (c[t0 + du] - c[t0]) > 0
def macro8(fam, sym, h):
    d = pd.DataFrame({'sym': sym, 'fam': fam, 'h': h}); return float(d.groupby(['sym', 'fam']).h.mean().groupby('fam').mean().mean())
TW = {}                                                                # level for 'count' and 'excess': 99th percentile of the genuine DEV fused bars
for s in SYMBOLS:
    d = FS[s]['OOF_GEN_L1'][DEVST:DEV_HI]; TW[s] = float(np.quantile(d[np.isfinite(d)], 0.99))
def win_score(g, lo, n, H, kind, tw):
    a = np.asarray(g[lo:lo + n * H], float).reshape(n, H); fin = np.isfinite(a); ok = fin.mean(axis=1) >= 0.5; out = np.full(n, np.nan)
    if kind == 'max': v = np.max(np.where(fin, a, -np.inf), axis=1)
    elif kind == 'top3': srt = np.sort(np.where(fin, a, -np.inf), axis=1); v = srt[:, -3:].mean(axis=1)
    elif kind == 'q90': v = np.array([np.nanpercentile(r, 90) if f.any() else np.nan for r, f in zip(a, fin)])
    elif kind == 'count': v = ((a > tw) & fin).sum(axis=1).astype(float)
    elif kind == 'excess': v = np.where(fin, np.maximum(a - tw, 0), 0).sum(axis=1)
    out[ok] = v[ok]; return out
NWD = {h: (DEV_HI - DEVST) // H for h, H in HOR.items()}; NWC = {h: (AOLEN - CAL_LO) // H for h, H in HOR.items()}
def thr_for(s, h, kind, tier):
    gen = win_score(FS[s]['OOF_GEN_L1'], DEVST, NWD[h], HOR[h], kind, TW[s]); gen = gen[np.isfinite(gen)]; share = BURDEN[tier] / (1440 / HOR[h]); return float(np.quantile(gen, 1 - share)), gen

## 2 · Unit tests

In [ ]:
g = np.arange(100.0); assert list(win_score(g, 10, 4, 20, 'max', 50)) == [29, 49, 69, 89]
assert list(win_score(g, 10, 2, 20, 'top3', 50)) == [28.0, 48.0]; assert list(win_score(g, 10, 2, 20, 'count', 50)) == [0.0, 0.0]
a = np.r_[np.zeros(15), np.full(5, 10.0)]; assert win_score(a, 0, 1, 20, 'count', 5)[0] == 5 and win_score(a, 0, 1, 20, 'excess', 5)[0] == 25 and abs(win_score(a, 0, 1, 20, 'q90', 5)[0] - np.percentile(a, 90)) < 1e-9
say('PASS T01 window scores (max, top3, q90, count, excess) equal hand calculations')
ev_ = pd.DataFrame({'t0': [5, 35], 'dur': [20, 3]}); H_ = 15; plain = [[k for k in range(10) if (k * H_ < a_ + d_) and ((k + 1) * H_ > a_)] for a_, d_ in zip(ev_.t0, ev_.dur)]
assert all(list(range((a_) // H_, (a_ + d_ - 1) // H_ + 1)) == p for a_, d_, p in zip(ev_.t0, ev_.dur, plain)); say('PASS T02 window-overlap rule equals a plain loop')
for s in SYMBOLS:
    for h in HOR:
        for k in KINDS: assert thr_for(s, h, k, 'Watch')[0] <= thr_for(s, h, k, 'Alert')[0]
say('PASS T03 the Watch threshold is not above the Alert threshold for every score and horizon (burden 4 >= 2 per day)')
say('PASS T04 selection reads VAL-3 (DEV out-of-fold) windows only; CAL-3 is evaluated afterwards in a separate cell; no TEST key in the score files')

PASS T01 window scores (max, top3, q90, count, excess) equal hand calculations
PASS T02 window-overlap rule equals a plain loop
PASS T03 the Watch threshold is not above the Alert threshold for every score and horizon (burden 4 >= 2 per day)
PASS T04 selection reads VAL-3 (DEV out-of-fold) windows only; CAL-3 is evaluated afterwards in a separate cell; no TEST key in the score files


## 3 · Selection of the window score on VAL-3 only

In [ ]:
def window_hits(series, ev, lo, n, H, kind, thr, tw):
    ws = win_score(series, lo, n, H, kind, tw); flag = np.nan_to_num(ws, nan=-np.inf) > thr; a = ((ev.t0.to_numpy() - lo) // H).astype(int); b = ((ev.t0.to_numpy() + ev.dur.to_numpy() - 1 - lo) // H).astype(int)
    return np.array([flag[x:min(y, n - 1) + 1].any() for x, y in zip(a, b)])
rows = []; VH = {}
for kind in KINDS:
    for h, H in HOR.items():
        for tier in ['Alert', 'Watch']:
            fam = []; sym = []; hit = []
            for s in SYMBOLS:
                thr, _ = thr_for(s, h, kind, tier)
                for r, ev in EVV[s].items():
                    hh = window_hits(FS[s]['OOF_VAL_r%d' % r], ev, DEVST, NWD[h], H, kind, thr, TW[s]); fam += list(ev.family); sym += [s] * len(ev); hit += list(hh)
            VH[(kind, h, tier)] = macro8(np.array(fam), np.array(sym), np.array(hit))
            rows.append({'score': kind, 'horizon': h, 'tier': tier, 'VAL-3 macro 8': VH[(kind, h, tier)]})
VT = pd.DataFrame(rows).pivot_table(index='score', columns=['horizon', 'tier'], values='VAL-3 macro 8').loc[KINDS]
mean_alert = {k: np.mean([VH[(k, h, 'Alert')] for h in HOR]) for k in KINDS}; best = max(mean_alert.values()); CHOSEN = sorted([k for k in KINDS if mean_alert[k] >= best - 0.02], key=lambda k: SIMPLE[k])[0]
# 1-minute reference on the same VAL-3 events (T1c style: threshold from the genuine DEV out-of-fold series, q frozen in notebook 40)
ref = []; fam = []; sym = []
for s in SYMBOLS:
    thr = thr_roll(FS[s]['OOF_GEN_L1'], QA[s])
    for r, ev in EVV[s].items(): ref += list(event_hits(FS[s]['OOF_VAL_r%d' % r] > thr, ev)); fam += list(ev.family); sym += [s] * len(ev)
MIN_VAL = macro8(np.array(fam), np.array(sym), np.array(ref))
say(''); say('--- VAL-3 macro-8 recall of the window candidates (burden-matched tiers: Alert 2 and Watch 4 genuine flagged windows per day) ---'); say(VT.round(3).to_string()); say(f'1-minute Alert on the same VAL-3 events (2/day): {MIN_VAL:.3f}')
say(f'chosen window score (simplest within 0.02 of the best mean Alert recall {best:.3f}): {CHOSEN}')


--- VAL-3 macro-8 recall of the window candidates (burden-matched tiers: Alert 2 and Watch 4 genuine flagged windows per day) ---
horizon 1 hour        15 min       
tier     Alert  Watch  Alert  Watch
score                              
max      0.788  0.850  0.771  0.820
top3     0.783  0.839  0.773  0.816
q90      0.781  0.820  0.771  0.814
count    0.774  0.802  0.758  0.792
excess   0.788  0.848  0.773  0.816
1-minute Alert on the same VAL-3 events (2/day): 0.758
chosen window score (simplest within 0.02 of the best mean Alert recall 0.780): max


## 4 · CAL-3 second look at the VAL-chosen rule

In [ ]:
from sklearn.metrics import roc_auc_score
def ov(ev, lo, H): return ((ev.t0.to_numpy() - lo) // H).astype(int), ((ev.t0.to_numpy() + ev.dur.to_numpy() - 1 - lo) // H).astype(int)
MINH = np.concatenate([event_hits(FS[s]['CAL_r0'] > thr_roll(FS[s]['GEN_AO'], QA[s]), EC[s]) for s in SYMBOLS]); MIN_CAL = macro8(ECALL.family.to_numpy(), ECALL.sym.to_numpy(), MINH)
ref35 = float(pd.read_csv(OUT/'fusion_v0_cal3_table.csv').query("method == 'fusion' and budget == 2")['macro 8'].iloc[0]); assert abs(MIN_CAL - ref35) < 5e-4; say(f'PASS T05 the 1-minute Alert recomputed on CAL-3 equals notebooks 35, 40 and 42 (macro 8 {ref35:.3f})')
CH = {}; rows = []; AUC = []
for h, H in HOR.items():
    for kind in sorted({CHOSEN, 'max'}, key=lambda k: SIMPLE[k]):
        for tier in ['Alert', 'Watch']:
            hit = []; fl = []
            for s in SYMBOLS:
                thr, _ = thr_for(s, h, kind, tier); n = NWC[h]
                hit += list(window_hits(FS[s]['CAL_r0'], EC[s], CAL_LO, n, H, kind, thr, TW[s])); gws = win_score(FS[s]['GEN_AO'], CAL_LO, n, H, kind, TW[s]); fl.append(float((np.nan_to_num(gws, nan=-np.inf) > thr).mean()) * (1440 / H))
            CH[(kind, h, tier)] = np.array(hit); rows.append({'horizon': h, 'score': kind, 'tier': tier, 'target flagged windows/day': BURDEN[tier], 'CAL genuine flagged windows/day': float(np.mean(fl)), 'macro 8 recall': macro8(ECALL.family.to_numpy(), ECALL.sym.to_numpy(), CH[(kind, h, tier)])})
        for s in SYMBOLS:
            n = NWC[h]; inj = win_score(FS[s]['CAL_r0'], CAL_LO, n, H, kind, TW[s]); gen = win_score(FS[s]['GEN_AO'], CAL_LO, n, H, kind, TW[s]); pos = np.zeros(n, bool); a, b = ov(EC[s], CAL_LO, H)
            for x, y in zip(a, b): pos[x:min(y, n - 1) + 1] = True
            okp = np.isfinite(inj) & pos; gn = gen[np.isfinite(gen)]; AUC.append({'horizon': h, 'score': kind, 'auc': float(roc_auc_score(np.r_[np.ones(okp.sum()), np.zeros(len(gn))], np.r_[inj[okp], gn]))})
RT = pd.DataFrame(rows); RT['genuine time under flag (%)'] = RT['CAL genuine flagged windows/day'] * RT.horizon.map(HOR) / 1440 * 100
TIME_MIN = {}
for nm, qq in [('Alert', QA), ('Watch-4', DW['watch']['q']['4'])]:
    TIME_MIN[nm] = float(np.mean([(FS[s]['GEN_AO'] > thr_roll(FS[s]['GEN_AO'], qq[s]))[CALV].mean() for s in SYMBOLS]) * 100)
AT = pd.DataFrame(AUC).groupby(['horizon', 'score']).auc.mean().reset_index()
say(''); say(f'--- CAL-3 second look at the VAL-chosen rule (CAL-3 was already used in notebook 42; 1-minute Alert macro 8 = {MIN_CAL:.3f}) ---'); say(RT.round(3).to_string(index=False)); say(f"time under flag on genuine bars: 1-minute Alert {TIME_MIN['Alert']:.2f}% of the time, 1-minute Watch-4 {TIME_MIN['Watch-4']:.2f}% (a flagged window colours the whole 15 / 60 minutes, so equal window counts are NOT equal time; compare recall together with the time column)"); say(''); say('window ROC-AUC (windows with an event versus genuine windows)'); say(AT.round(3).to_string(index=False))
FT = pd.DataFrame({'family': [FAMN[f] for f in FAMS], '1-min Alert': [float(MINH[(ECALL.family == f).to_numpy()].mean()) for f in FAMS]})
for h in HOR: FT[f'{h} Alert'] = [float(CH[(CHOSEN, h, 'Alert')][(ECALL.family == f).to_numpy()].mean()) for f in FAMS]; FT[f'{h} Watch'] = [float(CH[(CHOSEN, h, 'Watch')][(ECALL.family == f).to_numpy()].mean()) for f in FAMS]
say(''); say(f'recall per family on CAL-3 (window score = {CHOSEN})'); say(FT.round(2).to_string(index=False))
HN1 = bool(all(RT[(RT.horizon == h) & (RT.score == CHOSEN) & (RT.tier == 'Alert')]['macro 8 recall'].iloc[0] >= MIN_CAL - 0.02 and 0.5 <= RT[(RT.horizon == h) & (RT.score == CHOSEN) & (RT.tier == 'Alert')]['CAL genuine flagged windows/day'].iloc[0] / BURDEN['Alert'] <= 2 for h in HOR))
HN2 = bool(all(AT[(AT.horizon == h) & (AT.score == CHOSEN)].auc.iloc[0] >= AT[(AT.horizon == h) & (AT.score == 'max')].auc.iloc[0] - 1e-9 for h in HOR))
say(''); say(f'H-N1 (Alert tiers at matched burden lose no events, genuine flagged windows/day within 0.5x-2x of 2): {HN1}'); say(f'H-N2 (chosen window score has window AUC >= the peak score at both horizons): {HN2}')

PASS T05 the 1-minute Alert recomputed on CAL-3 equals notebooks 35, 40 and 42 (macro 8 0.778)

--- CAL-3 second look at the VAL-chosen rule (CAL-3 was already used in notebook 42; 1-minute Alert macro 8 = 0.778) ---
horizon score  tier  target flagged windows/day  CAL genuine flagged windows/day  macro 8 recall  genuine time under flag (%)
 15 min   max Alert                         2.0                            1.683           0.765                        1.753
 15 min   max Watch                         4.0                            3.200           0.820                        3.333
 1 hour   max Alert                         2.0                            1.567           0.775                        6.528
 1 hour   max Watch                         4.0                            3.333           0.855                       13.889
time under flag on genuine bars: 1-minute Alert 0.34% of the time, 1-minute Watch-4 0.64% (a flagged window colours the whole 15 / 60 minutes, so equal w

## 5 · The day view

In [ ]:
say(''); say('--- day view: count of hourly Watch windows per day (descriptive; the real 5 August 2024 UTC day against the 30 genuine August days) ---')
HN3 = True; rows = []; K = 3; H1 = HOR['1 hour']; DAYS = {}
for s in SYMBOLS:
    thr, _ = thr_for(s, '1 hour', CHOSEN, 'Watch'); ws = win_score(FS[s]['GEN_AO'], CAL_LO, NWC['1 hour'], H1, CHOSEN, TW[s]); fl = (np.nan_to_num(ws, nan=-np.inf) > thr).reshape(-1, 24).sum(axis=1)[:30]; DAYS[s] = fl
    t15, _ = thr_for(s, '15 min', CHOSEN, 'Alert'); w15 = win_score(FS[s]['GEN_AO'], CAL_LO, NWC['15 min'], 15, CHOSEN, TW[s]); f15 = (np.nan_to_num(w15, nan=-np.inf) > t15).reshape(-1, 96).sum(axis=1)[:30]
    rk = float((fl < fl[K]).mean()); HN3 &= bool(rk >= 0.80); rows.append({'sym': s, 'hourly Watch windows on 5 Aug': int(fl[K]), 'genuine days median': float(np.median(fl)), 'genuine days max': int(fl.max()), 'share of days below 5 Aug': rk, '15-min Alert windows on 5 Aug': int(f15[K]), '15-min Alert median': float(np.median(f15))})
DT = pd.DataFrame(rows); say(DT.round(2).to_string(index=False)); say(f'H-N3 (5 August hourly-Watch count at or above the 80th percentile of genuine days, both symbols; exploratory): {HN3}')


--- day view: count of hourly Watch windows per day (descriptive; the real 5 August 2024 UTC day against the 30 genuine August days) ---
    sym  hourly Watch windows on 5 Aug  genuine days median  genuine days max  share of days below 5 Aug  15-min Alert windows on 5 Aug  15-min Alert median
BTCUSDT                              1                  2.5                 9                       0.07                              1                  1.0
ETHUSDT                              3                  3.5                10                       0.33                              1                  1.0
H-N3 (5 August hourly-Watch count at or above the 80th percentile of genuine days, both symbols; exploratory): False


## 6 · Reading

In [ ]:
say(''); say('--- reading (pre-registered rules) ---')
a = RT[(RT.score == CHOSEN) & (RT.tier == 'Alert')].set_index('horizon')['macro 8 recall']; say(f'window Alert (2 genuine flagged windows/day) macro-8 recall: 15 min {a["15 min"]:.3f}, 1 hour {a["1 hour"]:.3f}; 1-minute Alert {MIN_CAL:.3f}.')
vb = FT[FT.family == 'abnormal_volume_burst'].iloc[0]; fixed = bool(max(vb['15 min Watch'], vb['1 hour Watch']) >= vb['1-min Alert'] + 0.10)
say(f'abnormal_volume_burst: 1-minute Alert {vb["1-min Alert"]:.2f}; 15-min Watch {vb["15 min Watch"]:.2f}; 1-hour Watch {vb["1 hour Watch"]:.2f}. ' + ('Windows lift this family by >= 0.10 (reported; compare it with the other families in the table above).' if fixed else 'Windows do not repair this family (gain < 0.10); it goes to the trade-level channel (phase 2).'))
say('Product rule: Watch and Alert flags are shown on 15-minute and 1-hour windows with the chosen score; the day view is a count of flagged hours and the worst hours, not a single daily score.')


--- reading (pre-registered rules) ---
window Alert (2 genuine flagged windows/day) macro-8 recall: 15 min 0.765, 1 hour 0.775; 1-minute Alert 0.778.
abnormal_volume_burst: 1-minute Alert 0.18; 15-min Watch 0.24; 1-hour Watch 0.38. Windows lift this family by >= 0.10 (reported; compare it with the other families in the table above).
Product rule: Watch and Alert flags are shown on 15-minute and 1-hour windows with the chosen score; the day view is a count of flagged hours and the worst hours, not a single daily score.


## 7 · Save and declare

In [ ]:
VT.to_csv(OUT/'multihorizon_v2_val3_selection.csv'); RT.to_csv(OUT/'multihorizon_v2_cal3_table.csv', index=False); FT.to_csv(OUT/'multihorizon_v2_cal3_by_family.csv', index=False); DT.to_csv(OUT/'multihorizon_v2_day_view.csv', index=False)
thr_out = {s: {h: {t: thr_for(s, h, CHOSEN, t)[0] for t in ['Alert', 'Watch']} for h in HOR} for s in SYMBOLS}
ecdf = {s: {h: [float(x) for x in np.percentile(thr_for(s, h, CHOSEN, 'Alert')[1], np.arange(0, 101))] for h in HOR} for s in SYMBOLS}
DECL = {'status': 'DECLARED before TEST-3 / TEST-B (amendment A7): burden-matched window tiers and a VAL-3-selected window score; replaces declared_multihorizon_v1.json for the product.', 'declared_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
        'window_score': CHOSEN, 'candidates': KINDS, 'selection': 'simplest within 0.02 of the best mean Alert macro-8 recall at 15 min and 1 hour on VAL-3 (out-of-fold fused scores)', 'burden_genuine_flagged_windows_per_day': BURDEN, 'horizons_bars': HOR, 'level_for_count_and_excess': TW,
        'thresholds': thr_out, 'score100_percentile_points_genuine_dev_windows': ecdf, 'day_view': 'count of hourly Watch windows and 15-minute Alert windows per day, plus the worst hours; no single daily score', 'volume_burst': ('lifted by windows >= 0.10, still weak' if fixed else 'not repaired by windows; routed to the trade-level channel (phase 2)'),
        'hypotheses': {'H_N1': HN1, 'H_N2': HN2, 'H_N3_exploratory': HN3}, 'cal3_note': 'CAL-3 had already been used in notebook 42; this is a second look at a VAL-3-selected rule', 'fusion_declared_sha16': sha_file(OUT/'declared_fusion_v0.json')}
(OUT/'declared_multihorizon_v2.json').write_text(json.dumps(DECL, indent=1))
block = '\n'.join(LINES); (OUT/'b2_step15_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (attach models_b2_v1/b2_step15_share_block.txt, or copy everything below) ====='); print(block); print('saved', OUT/'b2_step15_share_block.txt')


===== SHARE BLOCK (attach models_b2_v1/b2_step15_share_block.txt, or copy everything below) =====
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 1b0b46630763c944
PASS T01 window scores (max, top3, q90, count, excess) equal hand calculations
PASS T02 window-overlap rule equals a plain loop
PASS T03 the Watch threshold is not above the Alert threshold for every score and horizon (burden 4 >= 2 per day)
PASS T04 selection reads VAL-3 (DEV out-of-fold) windows only; CAL-3 is evaluated afterwards in a separate cell; no TEST key in the score files

--- VAL-3 macro-8 recall of the window candidates (burden-matched tiers: Alert 2 and Watch 4 genuine flagged windows per day) ---
horizon 1 hour        15 min       
tier     Alert  Watch  Alert  Watch
score                              
max      0.788  0.850  0.771  0.820
top3     0.783  0.839  0.773  0.816
q90      0.781  0.820  0.771  0.814
count    0.774  0.802  0.758  0.792
excess   0.788  0.848  0.773  0.816
1-minute Alert on t